<center>
    <h1>Probabilistic models taken by Storm</h1>
    <h2>Step 2: Model Checking</h2>
    <br>
    <br>
    <img src="images/storm_logo.png" width="500px"/>
    <h3>Matthias Volk</h3>
</center>

<div align="right">Press <em>spacebar</em> to navigate</div>


# Preparation
Load models from before

In [ ]:
# Import stormvogel and prism models from the previous step
import stormvogel
import stormpy

from examples.orchard_builder import build_simple, build_full, build_prism

In [ ]:
orchard_simple = build_simple()
orchard, orchard_storm = build_full()
orchard_prism, _ = build_prism()

In [ ]:
# Skip

# Analysis

## Questions for Orchard game
- *"What is the probability of winning?"*
- *"How long does a game take on average?"*

### Labels
- Use labels to identify states of interest
- Orchard Game has labels `PlayersWon` and `RavenWon`

### Rewards
- Allow to associate *reward* or *cost*
- Orchard Game has reward structure `rounds`

# Properties for Orchard
## Winning
- *What is the maximal probability that the players win the game?*

- $\mathbb{P}_{max=?} (F \; \text{PlayersWon})$

In [ ]:
prob_players_won = 'Pmax=? [F "PlayersWon"]'
result = stormvogel.model_checking(orchard_simple, prob_players_won)
print(result.get_result_of_state(orchard_simple.get_initial_state()))

In [ ]:
# Skip

## Winning (Visualization)
- *What is the maximal probability that the players win the game?*

In [ ]:
vis = stormvogel.show(orchard_simple, result=result)

In [ ]:
# Skip

# Properties for Orchard
## Expected duration
- *What is the maximal expected total number of rounds until the game ends?"*

- $\mathbb{R}^{\text{rounds}}_{max=?} (F\;( \text{PlayersWon} \vee \text{RavenWon}))$

In [ ]:
reward_prop = 'R{"rounds"}max=? [F "PlayersWon" | "RavenWon"]'
print(stormvogel.model_checking(orchard, reward_prop).get_result_of_state(orchard.get_initial_state()))
reward_prop = 'R{"rounds"}min=? [F "PlayersWon" | "RavenWon"]'
print(stormvogel.model_checking(orchard, reward_prop).get_result_of_state(orchard.get_initial_state()))

In [ ]:
# Skip

# Beyond
## Reward-bounded reachability
- *"What is the probability to win within $k$ rounds?"*
- $\mathbb{P}_{max=?} (F^{\text{rounds}\leq k}\;\text{PlayersWon})$

In [ ]:
# Helper function
def model_check(model, prop):
    formula = stormpy.parse_properties(prop)[0]
    result = stormpy.model_checking(model, formula, only_initial_states=True)
    return result.at(model.initial_states[0])

In [ ]:
# Winning probability within k rounds
probabilities = []
for k in range(41):
    win_steps = 'Pmax=? [F{"rounds"}<=' + str(k) + ' "PlayersWon"]'
    probabilities.append(model_check(orchard_prism, win_steps))

In [ ]:
# Skip

In [ ]:
# Plot
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (15,5)
plt.xlabel('Number of rounds')
plt.ylabel('Probability of players winning')
plt.title("Orchard")

# Plot results for all steps
plt.plot(range(len(probabilities)), probabilities, linewidth=2);

In [ ]:
# Skip

## Conditional reachability probability
- *"What is the maximal winning probability under the condition that the raven is eventually only one step away from the orchard?*

In [ ]:
query = 'Pmax=? [F "PlayersWon" || F "RavenOneAway"]'
print(model_check(orchard_prism, query))

In [ ]:
# Skip

## Multi-objective queries
- *"What is the maximal expected number of rounds of the game (1) among all policies that induce a winning probability of at least 60% (2)?*

In [ ]:
query = 'multi(R{"rounds"}max=? [F ("PlayersWon" | "RavenWon")], P>=0.60 [F "PlayersWon"])'
print(model_check(orchard_prism, query))

In [ ]:
# Skip